This script calculate:



1. 200+ RDKit descriptors

2. 1024-bit Morgan fingerprints (radius 2)

saves as separate .csv files for further modeling and EDA.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Install RDKit
!pip install rdkit

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors, AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
import os
import numpy as np
from rdkit import DataStructs


In [ ]:
# === Setup ===
input_path = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/processed-data/activity-labeled-data/"
output_path = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/descriptors"


In [ ]:
os.makedirs(output_path, exist_ok=True)

# List of cleaned files for 7 AD targets
target_files = {
    "bace1": "bace1_cleaned_labeled_smiles.csv",
    "buche": "buche_cleaned_labeled_smiles.csv",
    "ache": "ache_cleaned_labeled_smiles.csv",
    "gsk-3beta": "3beta_cleaned_labeled_smiles.csv",
    "mao-b": "mao-b_cleaned_labeled_smiles.csv",
    "esr1": "esr1_cleaned_labeled_smiles.csv",
    "5-HT6": "5-HT6_cleaned_labeled_smiles.csv"
}

In [ ]:
# RDKit descriptor functions
rdkit_descriptor_names = [desc[0] for desc in Descriptors._descList]
rdkit_descriptor_funcs = [desc[1] for desc in Descriptors._descList]

def calculate_rdkit_descriptors(mol):
    return [func(mol) for func in rdkit_descriptor_funcs]

In [ ]:
def calculate_morgan_fingerprint(mol, radius=2, nBits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
    arr = np.zeros((nBits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


In [ ]:
# === Loop through target files ===
for target, filename in target_files.items():
    file_path = os.path.join(input_path, filename)
    df = pd.read_csv(file_path)

    mols = [Chem.MolFromSmiles(smiles) for smiles in df['canonical_smiles']]

    # Filter out None molecules
    valid_indices = [i for i, mol in enumerate(mols) if mol is not None]
    mols = [mols[i] for i in valid_indices]
    df = df.iloc[valid_indices].reset_index(drop=True)

    # RDKit Descriptors
    rdkit_desc = [calculate_rdkit_descriptors(mol) for mol in mols]
    rdkit_df = pd.DataFrame(rdkit_desc, columns=[f"rdk_{name}" for name in rdkit_descriptor_names])

    # Morgan Fingerprints
    morgan_fp = [calculate_morgan_fingerprint(mol) for mol in mols]
    morgan_df = pd.DataFrame(morgan_fp, columns=[f"fp_{i}" for i in range(morgan_fp[0].shape[0])])

    # Combine
    feature_df = pd.concat([df[['molecule_chembl_id', 'canonical_smiles', 'pIC50']], rdkit_df, morgan_df], axis=1)
    feature_df.head()

    # Save
    output_file = os.path.join(output_path, f"{target}_features.csv")
    feature_df.to_csv(output_file, index=False)

Streaming output truncated to the last 5000 lines.
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:59:09] DEPRECATION WARNING: please use MorganGenerator
[10:5